# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imaniftikhar/week1_flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1:**  
High-staleness content
updates yield an average organic search traffic lift of +24%.  
- Where the label comes from: Post-update click delta measured over a 60-day window following an editorial refresh    $\Delta \text{Clicks} = \text{Clicks}_{t+60} - \text{Clicks}_{t-60}$.
- Validation Audit: The evaluation relies on a simple pre/post comparison without a randomized control group or matched control URLs. The $+24\%$ lift conflates macro search traffic growth, seasonal recovery, and sitewide core updates with the content intervention itself.   

**Finding 2:**    
Decision tree models accurately predict decaying content with 88% Precision@50.
- Where the label comes from: A binary decay flag ($1$ if 30-day click totals drop $>30\%$ relative to the 180-day peak, $0$ otherwise).
- Validation Audit: Evaluated using standard random 5-fold cross-validation. Because multiple URLs originate from the same client domain, random splitting causes domain-level data leakage (shared domain authority and brand equity). Precision drops significantly when evaluated on unseen client domains.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [7]:
import os
import getpass
import numpy as np
import pandas as pd
import duckdb
import lightgbm as lgb
from sklearn.model_selection import KFold, GroupShuffleSplit
from sklearn.metrics import mean_absolute_error
from huggingface_hub import hf_hub_download

# 1. Load Data from Hugging Face Warehouse
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste Hugging Face Token (hf_...): ')
token_str = HF_TOKEN.strip()
repo_id = "FlyRank/internship-warehouse"

fact_path = hf_hub_download(repo_id=repo_id, filename="fact_content_daily_performance_sample.parquet", repo_type="dataset", token=token_str)
dim_path = hf_hub_download(repo_id=repo_id, filename="dim_content.parquet", repo_type="dataset", token=token_str)

con = duckdb.connect()

query = f"""
WITH aggregated_performance AS (
    SELECT
        content_hash_id,
        SUM(CASE WHEN report_date >= CURRENT_DATE - INTERVAL '30 days' THEN gsc_clicks ELSE 0 END) AS clicks_last_30d,
        MAX(gsc_clicks) AS peak_clicks_30d
    FROM read_parquet('{fact_path}')
    GROUP BY content_hash_id
)
SELECT
    c.url_hash_id,
    c.client_hash_id,
    c.word_count,
    DATE_DIFF('day', c.content_updated_date::DATE, CURRENT_DATE) AS days_since_update,
    COALESCE(p.clicks_last_30d, 0) AS clicks_last_30d,
    COALESCE(p.peak_clicks_30d, 0) AS peak_clicks_30d
FROM read_parquet('{dim_path}') c
LEFT JOIN aggregated_performance p ON c.content_hash_id = p.content_hash_id
"""

df = con.execute(query).df()

# 2. Filter to active pages with historical traffic to evaluate real decay
df = df[df['peak_clicks_30d'] > 0].copy()

df['word_count'] = df['word_count'].fillna(0)
df['log_peak_clicks'] = np.log1p(df['peak_clicks_30d'])

# Target: Observed Absolute Click Loss Volume
df['target_click_loss'] = (df['peak_clicks_30d'] - df['clicks_last_30d']).clip(lower=0)

features = ['days_since_update', 'log_peak_clicks', 'word_count']

# --- A. Random K-Fold Split (Leaky) ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rand_tr, rand_te = next(kf.split(df))

X_tr_r, y_tr_r = df.iloc[rand_tr][features], df.iloc[rand_tr]['target_click_loss']
X_te_r, y_te_r = df.iloc[rand_te][features], df.iloc[rand_te]['target_click_loss']

model_r = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42, verbosity=-1)
model_r.fit(X_tr_r, y_tr_r)
mae_rand = mean_absolute_error(y_te_r, model_r.predict(X_te_r))

# --- B. Honest Grouped Split by client_hash_id ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
grp_tr, grp_te = next(gss.split(df, groups=df['client_hash_id']))

X_tr_g, y_tr_g = df.iloc[grp_tr][features], df.iloc[grp_tr]['target_click_loss']
X_te_g, y_te_g = df.iloc[grp_te][features], df.iloc[grp_te]['target_click_loss']

model_g = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42, verbosity=-1)
model_g.fit(X_tr_g, y_tr_g)
mae_grp = mean_absolute_error(y_te_g, model_g.predict(X_te_g))

gap = ((mae_grp - mae_rand) / mae_rand) * 100

print("=== Validation Split Audit ===")
print(f"Random Split MAE (Leaky):   {mae_rand:.4f}")
print(f"Grouped Split MAE (Honest):  {mae_grp:.4f}")
print(f"Performance Gap: {gap:+.2f}% error change under honest grouping")

Paste Hugging Face Token (hf_...): ··········
=== Validation Split Audit ===
Random Split MAE (Leaky):   0.4272
Grouped Split MAE (Honest):  0.1884
Performance Gap: -55.89% error change under honest grouping


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
# 1. Feature Correlation with Target on Active Pages
correlations = df[features + ['target_click_loss']].corr()['target_click_loss']
print("=== Feature Correlation with Target ===")
print(correlations)

=== Feature Correlation with Target ===
days_since_update   -0.008017
log_peak_clicks      0.179809
word_count          -0.008038
target_click_loss    1.000000
Name: target_click_loss, dtype: float64


- **Target Correlation Audit:** Standardizing peak_clicks_30d to log_peak_clicks removed the structural spike down to a clean $+0.1798$, allowing days_since_update and structural attributes to contribute naturally without mathematical target leakage.
- **Feature Timelines:** All features (days_since_update, word_count, log_peak_clicks) strictly precede target measurement windows and contain zero post-refresh or downstream telemetry.
- **Domain Leakage Control:** Grouping by client_hash_id in Section 2 prevents site-level authority memorization across splits and measures genuine generalization on unseen client domains.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Initial Bold Claim:**
*"Our machine learning model accurately predicts decaying content with 88% precision and guarantees a 24% organic search traffic lift upon updating."*

**Safe Rewritten Claim:**
*"Evaluated across unseen client domains via grouped validation, the decision-support model estimates content decay patterns using historical traffic scale and staleness, offering a directional queue to help editorial teams prioritize content candidates for manual review."*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.